# Survival analysis with a federated Cox model
In this notebook, you will study which patient characteristics are linked to survival.
You will use a **Cox model** (Cox proportional hazards model) with the data of the three hospitals.
The data stays at the hospitals: only intermediate results (sums and totals) go to the aggregator.

> **Important: first do the notebook `run-descriptive-statistics.ipynb`.**
> In that notebook, you explore the data of the hospitals. You will need what you learn there to finish this notebook.

At the end of this notebook, you have:
- run a federated Cox model with vantage6;
- read a hazard ratio and its confidence interval;
- explained what happens when organisations do not record their data in the same way and chosen what to do about it.


As before: read the text, run each cell from top to bottom (`Shift` + `Enter`) and look at the output.
Keep the developer network running (in case you stopped it, start it again: `python dev_network.py start`).

# What is a Cox model?
A Cox model looks at the **risk of passing away at any moment** during a certain time period, and how this risk is linked to patient characteristics (variables).

For each variable, the model gives a **hazard ratio**:
- a hazard ratio **above 1** means a **higher** risk of passing away;
- a hazard ratio **below 1** means a **lower** risk;
- a hazard ratio of exactly 1 means no difference.

For example, a hazard ratio of 1.5 for `clin_m` means that patients with metastases (`clin_m` is `1`) have a 1.5 times higher risk of passing away at any moment than patients without metastases.
For a number such as age, the hazard ratio is for each extra year.

The model also gives a **95 % confidence interval**: the range in which the true hazard ratio probably lies.
A wide interval means that we are less sure about the hazard ratio.

The model needs two more columns: the follow-up time (`overall_survival_in_days`) and the patient's vital status (`event_overall_survival`: `1` is deceased, `0` is alive).

# Connect and log in
This is the same as in the descriptive statistics notebook.

In [ ]:
from src.utils import check_server
from src.vantage_client import authenticate

VANTAGE6_CONFIG = {
    "server_url": "http://localhost",
    "server_port": 7601,
    "server_api": "/api",
}

check_server(VANTAGE6_CONFIG)
client = authenticate(VANTAGE6_CONFIG,
                      username="dev_admin",
                      password="password",
                      use_mfa=False,
                      use_encryption=False,
                      log_level="warn")
print("You are logged in.")

Run the next cell to check that you have done the descriptive statistics notebook.

In [ ]:
from src.cox import check_descriptive_statistics_done

check_descriptive_statistics_done(client)

# Choose the model
We start with a model with four variables:
- `age_at_diagnosis`: age in years;
- `performance_status_ecog`: how well the patient can do daily activities, from `0` (no problems) to `4` (in bed all day);
- `clin_m`: spread to other parts of the body, `0` (no) or `1` (yes);
- `clin_t`: the size of the tumour (T stage). We use it as a number: the hazard ratio is for each step up in T stage.

We use the data of all three organisations.
In a real study, you would choose the variables together with healthcare professionals and other domain experts, based on what is known about the disease.

We store every model in `models`, so that we can compare them later.

In [ ]:
from src.cox import run_cox_model

DATABASE_LABEL = "survival"

# The variables in the model; they must be numbers
MODEL_VARIABLES = ["age_at_diagnosis", "performance_status_ecog", "clin_m", "clin_t"]

# The organisations whose data is used
ORGANISATIONS_TO_RUN_TASK = [1, 2, 3]

models = {}

# Model A: all organisations, with T stage
Run the next cell.
The aggregator sends the question to the organisations several times, so this can take a minute or two.

In [ ]:
models["A: all organisations, with T stage"] = run_cox_model(
    client, ORGANISATIONS_TO_RUN_TASK, MODEL_VARIABLES, DATABASE_LABEL)

# What happened?
The model could not be calculated. Read the error message above.

Go back to the notebook `run-descriptive-statistics.ipynb`, to the section **Do the organisations use the same groups?**.
Look at the figure for `clin_t` and think about these questions:
1. Which organisation records the T stage differently?
2. Why can the model not use the T stage of this organisation as a number?

There are two simple ways to continue:
- **Option 1:** leave the T stage out of the model and keep all organisations.
- **Option 2:** keep the T stage in the model and leave out the organisation that records it differently.

Why not try both?

# Option 1 (model B): all organisations, without T stage
In this model, we remove `clin_t` from the list of variables.

In [ ]:
VARIABLES_WITHOUT_T_STAGE = ["age_at_diagnosis", "performance_status_ecog", "clin_m"]

models["B: all organisations, without T stage"] = run_cox_model(
    client, ORGANISATIONS_TO_RUN_TASK, VARIABLES_WITHOUT_T_STAGE, DATABASE_LABEL)

# Option 2 (model C): with T stage, without the organisation that records it differently
Now it is your turn.
**Change the list below**: remove the organisation that records the T stage differently.
Then run the cell.

If you still see an error, look again at the figure in the descriptive statistics notebook.

In [ ]:
# Change this list: leave out the organisation that records the T stage differently
ORGANISATIONS_WITH_SAME_CODING = [1, 2, 3]

models["C: same T stage coding, with T stage"] = run_cox_model(
    client, ORGANISATIONS_WITH_SAME_CODING, MODEL_VARIABLES, DATABASE_LABEL)

# Compare the models
The table below shows the Akaike Information Criterion (**AIC**) of each model.
The AIC tells you how well a model fits the data, with a penalty for each extra variable.
**A lower AIC is better.**

But be careful: **you can only compare the AIC of models that use exactly the same patients.**
Models B and C do not use the same patients, so you cannot compare their AIC.

To see what the T stage adds, we therefore make one more model (model D): with the same organisations as model C, but without the T stage.

In [ ]:
if models.get("C: same T stage coding, with T stage") is None:
    print("Model C is missing. Go back to option 2, change the list of organisations and run the cell again.")
else:
    models["D: same T stage coding, without T stage"] = run_cox_model(
        client, ORGANISATIONS_WITH_SAME_CODING, VARIABLES_WITHOUT_T_STAGE, DATABASE_LABEL)

In [ ]:
from src.cox import compare_models, show_hazard_ratios, plot_hazard_ratios

compare_models(models)

The next table and figure show the hazard ratio of each variable in each model, with the 95 % confidence interval between brackets.
In the figure, the dot is the hazard ratio, and the line is the confidence interval.
When the line crosses the dashed line (hazard ratio 1), we cannot be sure that the variable is linked to the risk of dying.

In [ ]:
show_hazard_ratios(models)

In [ ]:
plot_hazard_ratios(models)

# Question 1: what does removing the T stage do to the model?
Compare **model C** and **model D**. They use the same patients, so here you can compare the AIC.
- Which model has the lower AIC? Is the model better or worse without the T stage?
- Look at the hazard ratio of `clin_t` in model C. What does it tell you about the size of the tumour and the risk of dying?
- Do the hazard ratios of the other variables change when you remove the T stage? Why could that be? (Tip: in the descriptive statistics notebook, which hospital had older patients *and* larger tumours?)

# Question 2: is it better to leave out the organisation but keep the T stage?
Compare **model B** (all organisations, without T stage) and **model C** (fewer organisations, with T stage), but remember: you cannot compare their AIC.
- How many organisations does each model use? What does this do to the width of the confidence intervals?
- Which patients are missing from model C? Look back at the descriptive statistics notebook: how is this hospital different from the others (for example, age, tumour size, number of deaths)?
- Can the results of model C be used for patients like those of the missing hospital?
- Which model would you choose, and why? There is no single right answer: explain the advantages and disadvantages.

# Reflect
1. Both options lose information: option 1 loses a variable, option 2 loses patients.
Can you think of a third option that keeps both? (Tip: what could the organisation with groups such as `2a`, `2b` and `2c` do with them?)
2. Who should decide how data is recorded: each hospital, or all hospitals together? Why?
3. In a federated network, you cannot look at the data of the hospitals.
How did the descriptive statistics help you find the problem?

# When you have finished
Stop the developer network in the terminal with:
```shell
python dev_network.py stop
```

# All done?
Remove the developer network in the terminal with:
```shell
python dev_network.py remove
```